# 🖼️ Module 3 — Convolutional Neural Networks (CNNs)
### Introduction to Deep Learning | AgenticLabs.ng

---

## 🎯 Learning Objectives
- Understand why MLPs are inefficient for images
- Learn what convolution, pooling, and feature maps are
- Build a **LeNet-style CNN** from scratch in PyTorch
- Apply **transfer learning** using a pretrained ResNet-18
- Visualise what convolutional filters actually learn
- Train an image classifier on **CIFAR-10** (10 classes of objects)

---

## 3.1 — Why Not Just Use an MLP for Images?

A 224×224×3 RGB image has **150,528 pixels**. An MLP connecting that to 1024 hidden units needs ~**154 million parameters** in the first layer alone — mostly redundant, since nearby pixels are correlated.

CNNs solve this with **three key ideas**:
1. **Local connectivity** — each filter only looks at a small patch (e.g. 3×3)
2. **Weight sharing** — the same filter is applied everywhere in the image
3. **Pooling** — spatial downsampling to reduce computation


## 3.2 — What Does Convolution Do?

A convolutional filter (kernel) slides over the image and produces a **feature map** detecting a specific pattern (edge, curve, texture).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Manual convolution demo ────────────────────────────────
# Load a single MNIST image
from torchvision.datasets import MNIST
mnist = MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
img = mnist[0][0]  # shape: (1, 28, 28)

# Define hand-crafted filters
filters = {
    "Horizontal edge": torch.tensor([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=torch.float32),
    "Vertical edge":   torch.tensor([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=torch.float32),
    "Blur":            torch.ones(3,3, dtype=torch.float32) / 9,
    "Sharpen":         torch.tensor([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=torch.float32),
}

fig, axes = plt.subplots(1, len(filters)+1, figsize=(14, 3))
axes[0].imshow(img.squeeze(), cmap='gray'); axes[0].set_title("Original"); axes[0].axis('off')

for i, (name, f) in enumerate(filters.items()):
    conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
    with torch.no_grad():
        conv.weight.data = f.unsqueeze(0).unsqueeze(0)
    result = conv(img.unsqueeze(0)).squeeze().detach().numpy()
    axes[i+1].imshow(result, cmap='gray')
    axes[i+1].set_title(name, fontsize=10); axes[i+1].axis('off')

plt.suptitle("Effect of Different Convolution Filters", fontsize=12)
plt.tight_layout(); plt.show()


## 3.3 — Building a CNN for CIFAR-10

In [ ]:
# ── Load CIFAR-10 ─────────────────────────────────────────
CIFAR_CLASSES = ['airplane','automobile','bird','cat','deer',
                  'dog','frog','horse','ship','truck']

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010))
])

train_set = datasets.CIFAR10('./data', train=True,  download=True, transform=train_transform)
test_set  = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples : {len(train_set):,}")
print(f"Test samples     : {len(test_set):,}")

# Show samples
raw_set = datasets.CIFAR10('./data', train=True, transform=transforms.ToTensor())
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.ravel()):
    img, label = raw_set[i]
    ax.imshow(img.permute(1,2,0)); ax.set_title(CIFAR_CLASSES[label], fontsize=9); ax.axis('off')
plt.suptitle("CIFAR-10 Sample Images (32×32 RGB)", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── CNN Architecture ──────────────────────────────────────
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Block 1: 3 → 32 channels
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 32×32 → 16×16
            nn.Dropout(0.25)
        )
        # Block 2: 32 → 64 channels
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 16×16 → 8×8
            nn.Dropout(0.25)
        )
        # Block 3: 64 → 128 channels
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 8×8 → 4×4
            nn.Dropout(0.25)
        )
        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x)

model = CNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {total_params:,}")


In [ ]:
# ── Training loop ─────────────────────────────────────────
def train_epoch(model, loader, criterion, optimiser):
    model.train()
    total_loss = correct = total = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimiser.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimiser.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()
        total   += y.size(0)
    return total_loss / len(loader), correct/total*100

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            total_loss += criterion(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
            total   += y.size(0)
    return total_loss / len(loader), correct/total*100

criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=15)

train_losses, test_losses = [], []
train_accs, test_accs     = [], []

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Loss':>9} | {'Test Acc':>8}")
print("-" * 55)

for epoch in range(1, 16):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimiser)
    te_loss, te_acc = eval_epoch(model, test_loader, criterion)
    scheduler.step()
    
    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);   test_accs.append(te_acc)
    print(f"{epoch:5d} | {tr_loss:10.4f} | {tr_acc:8.2f}% | {te_loss:9.4f} | {te_acc:7.2f}%")


In [ ]:
# ── Plot training curves ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, 16)

ax1.plot(epochs, train_losses, 'b-o', ms=5, label='Train')
ax1.plot(epochs, test_losses, 'r-o',  ms=5, label='Test')
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, train_accs, 'b-o', ms=5, label='Train')
ax2.plot(epochs, test_accs, 'r-o',  ms=5, label='Test')
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy Curves"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f"CNN on CIFAR-10 — Best Test Acc: {max(test_accs):.2f}%", fontsize=13)
plt.tight_layout(); plt.show()


## 3.4 — Transfer Learning with ResNet-18

Instead of training from scratch, we can use a model pre-trained on ImageNet (1.2M images, 1000 classes) and fine-tune it for CIFAR-10. This is **transfer learning** — one of the most powerful techniques in practice.

In [ ]:
# ── Transfer Learning: ResNet-18 ──────────────────────────
# Load pretrained ResNet-18
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all layers except the final classifier
for param in resnet.parameters():
    param.requires_grad = False

# Replace final layer for 10 classes
resnet.fc = nn.Sequential(
    nn.Linear(resnet.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 10)
)

# Only fine-tune the new layers
resnet = resnet.to(device)
opt_resnet = optim.Adam(resnet.fc.parameters(), lr=0.001)

print("ResNet-18 final layer replaced for 10 classes")
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} (out of {sum(p.numel() for p in resnet.parameters()):,})")

# Quick training (5 epochs — notice how fast it converges!)
print(f"\n{'Epoch':>5} | {'Train Acc':>9} | {'Test Acc':>8}")
print("-" * 30)
for epoch in range(1, 6):
    _, tr_acc = train_epoch(resnet, train_loader, criterion, opt_resnet)
    _, te_acc = eval_epoch(resnet,  test_loader,  criterion)
    print(f"{epoch:5d} | {tr_acc:8.2f}% | {te_acc:7.2f}%")

print("\n✅ Transfer learning achieves strong accuracy in just 5 epochs!")


## 📝 Exercises

1. **Filter visualisation**: After training, extract `model.block1[0].weight.data` and plot the 32 learned filters. What patterns do you see?
2. **Architecture search**: Add a 4th convolutional block. Does it improve accuracy or cause overfitting?
3. **Data augmentation**: Add `transforms.RandomRotation(15)` to `train_transform`. How does it affect overfitting?
4. **Unfreeze ResNet**: After 5 epochs, unfreeze all ResNet layers and fine-tune with a very small lr (e.g. 1e-5). How much does accuracy improve?

---

## ✅ Module 3 Summary

| Concept | Key takeaway |
|---|---|
| Convolution | Learnable filter that slides over input, detecting local patterns |
| Feature map | Output of applying one filter to the entire input |
| MaxPool | Reduces spatial size, makes detection position-invariant |
| BatchNorm | Normalises layer inputs — stabilises and accelerates training |
| Transfer learning | Use pretrained weights, replace only the final classification layer |

**Next up → Module 4: Recurrent Networks & Sequence Modelling**
